# agentic_alpha_cross_asset_rotation_search_v1

Diagnostic R-G-E-V-D card for cross-asset leveraged ETF rotation. The hypothesis comes from sector-rotation, global tactical asset allocation, cross-asset momentum, and macro regime research: rotate among SOXL/TQQQ/TECL/SPXL/UPRO or throttle SOXL when broad leveraged-equity momentum weakens. The goal gate remains unchanged: beat `production/qlib_paper_sim_v1` while holding MaxDD above -20%.


In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.qlib_native import QlibProviderConfig, build_qlib_store_from_frame, load_qlib_feature_frame
from src.factor import build_cross_asset_price_frame, evaluate_cross_asset_rotation_candidates, summarize_cross_asset_branch
from src.workflow import load_agentic_alpha_backtest_metric

EXPERIMENT = 'agentic_alpha_cross_asset_rotation_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl_cross_asset_rotation_v1').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
PRODUCTION_CARD = ROOT / 'production' / 'qlib_paper_sim_v1'
RAW_SYMBOLS = ROOT / 'data' / 'raw' / 'symbols'
SYMBOLS = ['SOXL', 'TQQQ', 'TECL', 'SPXL', 'UPRO']
START = '2019-11-01'
SOURCE_END = '2026-04-15'
MAX_DRAWDOWN_FLOOR = -20.0

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
if PROVIDER_URI.exists():
    shutil.rmtree(PROVIDER_URI)

baseline = load_agentic_alpha_backtest_metric(PRODUCTION_CARD, window='test')

workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: diagnostic_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  cross_asset_rotation:
    start: {START}
    source_end: {SOURCE_END}
rgevd:
  source: https://github.com/JacobDu/agentic-alpha
  stages:
    - research
    - retrieve
    - generate
    - evaluate
    - validate
    - distill
goal_gate:
  baseline_card: production/qlib_paper_sim_v1
  baseline_return_pct: {baseline.return_pct}
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  require_return_above_baseline: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')

external_sources = pd.DataFrame(
    [
        {
            'source_type': 'github',
            'title': 'agentic-alpha R-G-E-V-D workflow',
            'url': 'https://github.com/JacobDu/agentic-alpha',
            'extracted_signal': 'Use retrieve-generate-evaluate-validate-distill loops to close failed strategy branches with evidence.',
            'retrieval_status': 'workflow_reference',
        },
        {
            'source_type': 'github',
            'title': 'Quant Sector Rotation Strategy',
            'url': 'https://github.com/garroshub/Quant_Sector_Rotation_Strategy',
            'extracted_signal': 'Combines momentum signals, volatility normalization, VIX sizing, and drawdown controls for ETF sector rotation.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'arxiv',
            'title': 'Tactical Asset Allocation with Macroeconomic Regime Detection',
            'url': 'https://arxiv.org/html/2503.11499v1',
            'extracted_signal': 'Use regime-aware forecasts and portfolio sizing rather than single-asset timing alone.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Global Tactical Cross-Asset Allocation',
            'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1276616',
            'extracted_signal': 'Cross-asset value and momentum can add tactical allocation value across asset classes.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Momentum Strategies Across Asset Classes',
            'url': 'https://www.cmegroup.com/education/files/jpm-momentum-strategies-2015-04-15-1681565.pdf',
            'extracted_signal': 'Prototype absolute and relative momentum filters, lookback windows, rebalancing, and risk controls are core design dimensions.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'paper',
            'title': 'Momentum Based Volatility Targeting',
            'url': 'https://quantpedia.com/how-to-faster-enhance-strategic-asset-allocation-with-tactical-models/',
            'extracted_signal': 'Combine momentum selection with volatility-targeted portfolios over multiple horizons.',
            'retrieval_status': 'web_retrieved',
        },
        {
            'source_type': 'twitter',
            'title': 'VolatilityVIX tactical rotation seed',
            'url': 'https://x.com/VolatilityVIX/status/1917586057316687929',
            'extracted_signal': 'Tactical rotation ideas should diversify signals instead of depending on a single trend rule.',
            'retrieval_status': 'search_seed',
        },
    ]
)
external_sources.to_csv(CARD / 'external_sources.csv', index=False)

raw_frames = {symbol: pd.read_csv(RAW_SYMBOLS / f'{symbol}.csv') for symbol in SYMBOLS}
raw_price_frame = build_cross_asset_price_frame(raw_frames, start=START, end=SOURCE_END)
qlib_rows = []
for symbol in SYMBOLS:
    symbol_frame = raw_price_frame[['datetime', f'{symbol}_close', f'{symbol}_volume']].rename(
        columns={f'{symbol}_close': 'close', f'{symbol}_volume': 'volume'}
    )
    symbol_frame['instrument'] = symbol
    qlib_rows.append(symbol_frame[['instrument', 'datetime', 'close', 'volume']])
qlib_store_frame = pd.concat(qlib_rows, ignore_index=True)
provider_manifest = build_qlib_store_from_frame(
    qlib_store_frame,
    provider_uri=PROVIDER_URI,
    market='all',
    region='us',
    freq='day',
    field_map={'close': 'close', 'volume': 'volume'},
    as_of=SOURCE_END,
    source='data/raw/symbols/SOXL,TQQQ,TECL,SPXL,UPRO.csv',
)
shutil.copyfile(PROVIDER_URI / 'qlib_store_manifest.json', CARD / 'qlib_store_manifest.json')

provider_config = QlibProviderConfig(provider_uri=PROVIDER_URI, region='us', market='all', freq='day')
qlib_feature_frame = load_qlib_feature_frame(
    provider_config,
    instruments=SYMBOLS,
    fields=['$close', '$volume'],
    start=raw_price_frame['datetime'].min().date().isoformat(),
    end=raw_price_frame['datetime'].max().date().isoformat(),
)
qlib_reset = qlib_feature_frame.reset_index()
qlib_price_frame = pd.DataFrame({'datetime': sorted(qlib_reset['datetime'].unique())})
for symbol in SYMBOLS:
    inst = qlib_reset.loc[qlib_reset['instrument'] == symbol, ['datetime', '$close', '$volume']].rename(
        columns={'$close': f'{symbol}_close', '$volume': f'{symbol}_volume'}
    )
    qlib_price_frame = qlib_price_frame.merge(inst, on='datetime', how='inner')
qlib_price_frame = qlib_price_frame.sort_values('datetime').reset_index(drop=True)

candidate_metrics = evaluate_cross_asset_rotation_candidates(
    qlib_price_frame,
    baseline_return_pct=baseline.return_pct,
    max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    symbols=SYMBOLS,
    rotation_lookbacks=[126, 252],
    breadth_lookbacks=[63, 126],
    caps=[0.25, 1.0, 2.0],
    breadth_scales=[0.5, 1.0, 2.0],
)
candidate_metrics.to_csv(CARD / 'agentic_alpha_cross_asset_rotation_candidate_metrics.csv', index=False)
branch_results = {
    branch: summarize_cross_asset_branch(
        candidate_metrics,
        branch=branch,
        baseline_return_pct=baseline.return_pct,
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
    )
    for branch in sorted(candidate_metrics['branch'].unique())
}
passes_both = candidate_metrics.loc[candidate_metrics['beats_baseline_return'] & candidate_metrics['passes_drawdown_gate']]
best_candidate = None if passes_both.empty else str(passes_both.sort_values('return_pct', ascending=False).iloc[0]['candidate'])

search_summary = {
    'workflow': 'agentic_alpha_cross_asset_rotation_search',
    'stage_order': ['research', 'retrieve', 'generate', 'evaluate', 'validate', 'distill'],
    'baseline': {
        'name': baseline.name,
        'return_pct': float(baseline.return_pct),
        'max_drawdown_pct': float(baseline.max_drawdown_pct),
        'benchmark_return_pct': float(baseline.benchmark_return_pct),
        'recorder_id': baseline.recorder_id,
    },
    'goal_achieved': bool(best_candidate is not None),
    'best_candidate': best_candidate,
    'passes_both_count': int(len(passes_both)),
    'branch_results': branch_results,
    'external_source_counts': external_sources['source_type'].value_counts().sort_index().to_dict(),
    'cross_asset_data': {
        'symbols': SYMBOLS,
        'source_coverage_start': raw_price_frame.attrs['source_coverage_start'],
        'source_coverage_end': raw_price_frame.attrs['source_coverage_end'],
        'candidate_coverage_start': candidate_metrics['coverage_start'].min(),
        'candidate_coverage_end': candidate_metrics['coverage_end'].max(),
        'provider_manifest': 'data/qlib/soxl_cross_asset_rotation_v1/qlib_store_manifest.json',
        'feature_rows': int(len(qlib_price_frame)),
        'qlib_audit_rows': int(len(qlib_reset)),
        'qlib_fields': ['$close', '$volume'],
    },
    'provider_manifest_hash': provider_manifest['hash'],
}
(CARD / 'agentic_alpha_cross_asset_rotation_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'failure_modes.md').write_text(
    '# Failure Modes\n\n'
    '- relative_momentum_rotation: cross-asset momentum can raise return versus lower-risk variants, but the high-return settings still accept deep leveraged-ETF drawdowns; drawdown-gated settings are too low exposure to approach the production baseline.\n'
    '- cross_asset_breadth_soxl: broad leveraged-equity confirmation does not create a MaxDD<20% SOXL path; all representative breadth candidates breach the drawdown gate.\n'
    '- coverage: local TQQQ/TECL/SPXL/UPRO raw symbol files start at 2019-11-01 and end at 2026-04-15, so this card cannot claim full 2026-06-09 production-window completion.\n'
    '- next_action: pivot from leveraged-equity rotation to either longer independent hedge assets, options-risk data, or model families that predict crash avoidance without simply lowering exposure.\n',
    encoding='utf-8',
)

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder_id = recorder.id
    R.log_params(
        baseline=baseline.name,
        provider_uri=PROVIDER_URI.as_posix(),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        source_coverage_end=raw_price_frame.attrs['source_coverage_end'],
        candidate_rows=len(candidate_metrics),
    )
    search_summary['recorder_id'] = recorder_id
    (CARD / 'agentic_alpha_cross_asset_rotation_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')
    for path in [
        CARD / 'agentic_alpha_cross_asset_rotation_candidate_metrics.csv',
        CARD / 'agentic_alpha_cross_asset_rotation_search_summary.json',
        CARD / 'external_sources.csv',
        CARD / 'failure_modes.md',
        CARD / 'qlib_store_manifest.json',
    ]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_cross_asset_rotation')
    recorder_local_dir = Path(recorder.get_local_dir())

for transient_name in ['code_cached.txt', 'code_diff.txt', 'code_status.txt']:
    transient_path = recorder_local_dir / 'artifacts' / transient_name
    if transient_path.exists():
        transient_path.unlink()
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
search_summary['recorder_artifacts'] = recorder_artifacts
(CARD / 'agentic_alpha_cross_asset_rotation_search_summary.json').write_text(json.dumps(search_summary, indent=2, sort_keys=True), encoding='utf-8')

summary = {
    'experiment': EXPERIMENT,
    'card_type': 'diagnostic_search',
    'promotion_decision': 'not_promoted' if not search_summary['goal_achieved'] else 'promote_candidate',
    'as_of': SOURCE_END,
    'data_range': [START, SOURCE_END],
    'provider_uri': PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'recorder_artifacts': recorder_artifacts,
    'market': 'all',
    'region': 'us',
    'baseline': search_summary['baseline'],
    'goal_gate': {'max_drawdown_floor': MAX_DRAWDOWN_FLOOR, 'require_return_above_baseline': True},
    'branch_results': branch_results,
    'search_report': search_summary,
    'external_source_counts': search_summary['external_source_counts'],
    'cross_asset_data': search_summary['cross_asset_data'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'model': 'https://qlib.readthedocs.io/en/latest/component/model.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start', 'build_qlib_store_from_frame'],
    'workflow_config': 'workflow.yaml',
    'sidecars': {
        'candidate_metrics': 'agentic_alpha_cross_asset_rotation_candidate_metrics.csv',
        'external_sources': 'external_sources.csv',
        'failure_modes': 'failure_modes.md',
        'qlib_store_manifest': 'qlib_store_manifest.json',
        'search_summary': 'agentic_alpha_cross_asset_rotation_search_summary.json',
    },
    'artifact_policy': {
        'policy': 'slim_card_recorder_first',
        'workflow_config': 'workflow.yaml',
        'machine_entrypoint': 'summary.json',
        'primary_artifact': f'{EXPERIMENT}.ipynb',
        'kept_sidecars': [
            'agentic_alpha_cross_asset_rotation_candidate_metrics.csv',
            'agentic_alpha_cross_asset_rotation_search_summary.json',
            'external_sources.csv',
            'failure_modes.md',
            'qlib_store_manifest.json',
        ],
        'recorder_only_artifacts': {
            'predictions': 'artifacts/pred.pkl',
            'labels': 'artifacts/label.pkl',
            'signals': 'artifacts/signals.pkl',
            'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
            'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
            'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
            'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        },
        'reproducible_from_recorder': True,
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
(CARD / 'conclusion.md').write_text(
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，把 Github/论文/Arxiv/Twitter 的跨资产动量和 TAA 思路映射到 SOXL/TQQQ/TECL/SPXL/UPRO，构建 qlib provider 后通过 `D.features` 读取并测试 relative momentum rotation 与 cross-asset breadth SOXL 两条分支。\n'
    f'- key_metrics: production baseline return={baseline.return_pct:.2f}%, maxDD={baseline.max_drawdown_pct:.2f}%; 本卡 passes_both_count={len(passes_both)}。\n'
    f'- regime_window_difference: 多资产 raw symbol 共同覆盖到 {raw_price_frame.attrs["source_coverage_end"]}，候选有效回测到 {candidate_metrics["coverage_end"].max()}，不能冒充完整 2026-06-09 生产窗口。\n'
    '- promotion_decision: not_promoted，未找到同时满足收益超过现有生产策略且 MaxDD 小于 20% 的候选。\n'
    '- risks: 轮动宇宙仍全是高相关 3x 股指/行业 ETF，风险预算降低回撤时同步削弱复利，不能提供真正独立的崩盘对冲。\n'
    '- next_action: 转向更长历史的独立对冲资产、期权风险曲面或新模型家族，不再重复同类 leveraged-equity rotation。\n',
    encoding='utf-8',
)
print(json.dumps({'recorder_id': recorder_id, 'passes_both_count': int(len(passes_both)), 'branch_results': branch_results}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[72565:MainThread](2026-06-23 17:33:08,351) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72565:MainThread](2026-06-23 17:33:08,354) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72565:MainThread](2026-06-23 17:33:08,354) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_cross_asset_rotation_v1')}


[72565:MainThread](2026-06-23 17:33:08,360) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[72565:MainThread](2026-06-23 17:33:09,939) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_cross_asset_rotation_search_v1.


[72565:MainThread](2026-06-23 17:33:09,943) INFO - qlib.workflow - [exp.py:258] - Experiment 356799567282419261 starts running ...


[72565:MainThread](2026-06-23 17:33:10,013) INFO - qlib.workflow - [recorder.py:345] - Recorder 8bb24b382c7c4634ac4e830f314fcadd starts running under Experiment 356799567282419261 ...


[72565:MainThread](2026-06-23 17:33:10,089) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "branch_results": {
    "cross_asset_breadth_soxl": {
      "best_drawdown_gate_available": false,
      "best_drawdown_gate_candidate": "lb63_agree2_scale2.0_volcap0.8",
      "best_drawdown_gate_max_drawdown_pct": -68.36314067096434,
      "best_drawdown_gate_return_pct": 731.6764013071153,
      "best_return_candidate": "lb63_agree2_scale2.0_volcap0.8",
      "best_return_max_drawdown_pct": -68.36314067096434,
      "best_return_pct": 731.6764013071153,
      "coverage_end": "2026-04-14",
      "coverage_start": "2019-11-01",
      "goal_baseline_return_pct": 6570.0232821043,
      "grid_rows": 120,
      "max_drawdown_floor": -20.0,
      "passes_both_count": 0
    },
    "relative_momentum_rotation": {
      "best_drawdown_gate_available": true,
      "best_drawdown_gate_candidate": "lb126_top3_abs-0.05_equal_cap0.25",
      "best_drawdown_gate_max_drawdown_pct": -19.85935030860453,
      "best_drawdown_gate_return_pct": 111.56311642743492,
      "best_return_candidate": "lb12